In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from collections import Counter
import ast
import re
import matplotlib.gridspec as gridspec
from itertools import product

from prophet import Prophet
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import colors as mcolors
from scipy.stats import linregress
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.cluster import KMeans
from tabulate import tabulate
from collections import Counter

%matplotlib inline


In [ ]:
def train_prophet(data, duration):
    df_all_amount = data[['Description', 'YearMonth', 'LEVEL_1', 'Pendapatan in IDR']].copy()
    cust_type_unique = df_all_amount['Description'].unique()
    prod_level1_unique = df_all_amount['LEVEL_1'].unique()

    combinations = pd.DataFrame(
        product(cust_type_unique, prod_level1_unique),
        columns=['Description', 'LEVEL_1']
    )
    start_forecast = df_all_amount['YearMonth'].max() + pd.DateOffset(months=1)
    forecast_dates = pd.date_range(start=start_forecast, periods=duration, freq='MS')
    df_forecast_tmp = pd.DataFrame({'YearMonth': forecast_dates})

    df_forecast_tmp = df_forecast_tmp.merge(pd.DataFrame(combinations), how='cross')


    df_all_amount.rename(columns={'Pendapatan in IDR': 'y', 'YearMonth': 'ds'}, inplace=True)
    df_forecast_tmp.rename(columns={'Pendapatan in IDR': 'y', 'YearMonth': 'ds'}, inplace=True)


    model = Prophet()
    regressor=[]
    if(len(cust_type_unique)>1):
        regressor.append('Description')
    if(len(prod_level1_unique)>1):
        regressor.append('LEVEL_1')

    for i in regressor:
        model.add_regressor(i)
    
    target_col_train=['ds','y',]+regressor
    target_col_forecast=['ds']+regressor
    model.fit(df_all_amount[target_col_train])

    # pred = model.predict(df_all_amount[target_col_forecast])
    pred_forecast = model.predict(df_forecast_tmp[target_col_forecast])


    # df_all_amount['yhat'] = pred['yhat']
    # df_forecast_tmp['yhat'] = pred_forecast['yhat']

    # df_all_amount['yhat'] = pred['yhat'].apply(lambda x: 0 if x < 0 else x)
    df_forecast_tmp['yhat'] = pred_forecast['yhat'].apply(lambda x: 0 if x < 0 else x)


    # combined_forecast = pd.concat([df_all_amount[['ds','yhat','Description', 'LEVEL_1']],
    #                                df_forecast_tmp[['ds','yhat','Description', 'LEVEL_1']]], ignore_index=True)
    combined_forecast = df_forecast_tmp[['ds','yhat','Description', 'LEVEL_1']]
    combined_forecast['Method'] = 'PROPHET'
    combined_forecast.rename(columns={'yhat': 'Amount_Predict'}, inplace=True)
    model_data = {
        'model': model,
        'scalers_numerical_train': None,
        'scalers_categorical_train': None,
        'scalers_log_col': None,
        'coefficients_df_sorted': None
    }
    results=combined_forecast[['ds','Description', 'LEVEL_1','Amount_Predict','Method']].merge(df_all_amount,on=['ds','Description', 'LEVEL_1'],how='left')  
    results.rename(columns={'ds':'YearMonth','y':'Pendapatan in IDR'}, inplace=True)

    return results[['YearMonth','Description', 'LEVEL_1','Pendapatan in IDR','Amount_Predict','Method']], model_data

In [ ]:
data_pendapatan=pd.read_csv('../data/data_pendapatan_cleared_mei_november_2023.csv').drop(columns=['Unnamed: 0'])
data_pendapatan.drop(columns=['Kontribusi in IDR','Kontribusi in USD','Pendapatan in USD'],inplace=True)


In [ ]:
cust_desc=pd.read_excel('./../data/customer_description_mapping.xlsx').drop(columns=['Unnamed: 0'])
cust_desc

In [ ]:
data_pendapatan['Tanggal Nota'] = pd.to_datetime(data_pendapatan['Tanggal Nota'])

In [ ]:
data_pendapatan['YearMonth'] = data_pendapatan['Tanggal Nota'].dt.to_period('M')

In [ ]:
data_pendapatan['YearMonth'].unique()

In [ ]:
data_pendapatan_used=data_pendapatan.loc[data_pendapatan['Pendapatan in IDR']>0]

In [ ]:
customer_data = data_pendapatan_used.groupby(['Cust No.','YearMonth','LEVEL_1'])[['Pendapatan in IDR']].sum()
customer_data.reset_index(inplace=True)

In [ ]:
customer_data.columns

In [ ]:
len(customer_data.loc[customer_data['Pendapatan in IDR']==0])

In [ ]:
len(customer_data['Cust No.'].unique())

In [ ]:
merged_data = customer_data.merge(cust_desc, how='left', right_on='Customer', left_on='Cust No.')
merged_data['Customer'].fillna(0,inplace=True)
merged_data['Customer']=merged_data['Customer'].astype('int64')
merged_data

In [ ]:
merged_data['Description'].fillna('Unknown', inplace=True)
merged_data

In [ ]:
merged_data.drop(columns=['Customer'],inplace=True)

In [ ]:
merged_data.to_excel('./../data/trans_customer_permonth_pertypedesc.xlsx')

In [ ]:
agg_type_cust = merged_data.groupby(['Description','YearMonth','LEVEL_1'])[['Pendapatan in IDR']].sum()
agg_type_cust.reset_index(inplace=True)
agg_type_cust

In [ ]:
agg_type_cust['Description'].unique()

In [ ]:
agg_type_cust.columns

In [ ]:
agg_type_cust_labeled=agg_type_cust.copy()

In [ ]:
label_encoder = LabelEncoder()
agg_type_cust_labeled['Description'] = label_encoder.fit_transform(agg_type_cust_labeled['Description'])

In [ ]:
label_encoder_level_1 = LabelEncoder()
agg_type_cust_labeled['LEVEL_1'] = label_encoder_level_1.fit_transform(agg_type_cust_labeled['LEVEL_1'])

In [ ]:
agg_type_cust_labeled

# Per Level_1

In [ ]:
agg_type_cust_labeled['Description'].value_counts()

In [ ]:
# Afiliasi -> 0
agg_type_cust_labeled_test=agg_type_cust_labeled.loc[agg_type_cust_labeled['Description']==0]

In [ ]:
agg_type_cust_labeled_test['YearMonth'] = agg_type_cust_labeled_test['YearMonth'].astype(str) + '-01'
agg_type_cust_labeled_test['YearMonth'] = pd.to_datetime(agg_type_cust_labeled_test['YearMonth'])
agg_type_cust_labeled_test

In [ ]:
all_data_test = pd.DataFrame()
all_model_data = {}
results, model_data = train_prophet(agg_type_cust_labeled_test, 3)

In [ ]:
results

In [ ]:
result_prod_recommended = results.groupby(['YearMonth','LEVEL_1'])[['Amount_Predict']].sum()
result_prod_recommended.reset_index(inplace=True)
result_prod_recommended = results.groupby(['LEVEL_1'])[['Amount_Predict']].mean()
result_prod_recommended.reset_index(inplace=True)
result_prod_recommended.sort_values(by=['Amount_Predict'],ascending=False)

In [ ]:
# Level 1
# flow predictions 
# kmeans :rfm feature + customer type 
# kmeans output -> cluster
# based on cluster -> top n product level 1
# input prophet : customer_type 
# prophet -> output : top n product 

# final output : take intersect between k means and result


# Level 1
# Cluster -> top trans on each type of level 1 > take top 3
# prophet -> top3 -> take intersect between k means and result

# Level 2 
# 
 